# Tworzenie datasetu do wykorzystania dla modelu

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import cv2

In [2]:
LOAD_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/GDSAM_172")
LOAD_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR = Path("/mnt/d/Backup/MAGISTERSKIE/outputs/MODEL")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

Pobranie odpowiednich danych

In [3]:
masks_array = np.load(str(LOAD_DIR)+'/masks_array.npy')
incomplete_detect = np.load(str(LOAD_DIR)+'/incomplete_detect.npy')
df = pd.read_csv(str(LOAD_DIR)+'/df.csv')

Znalezienie środków dla każdej maski/elementu chwytaka

In [4]:
xyz_array = []

for i in range(0, masks_array.shape[0]):

  # Załadowanie obrazu głębokości
  depth_img = Image.open(df['depth_path'][i]).convert("RGB")
  depth_img = np.asarray(depth_img)
  depth_img = depth_img[:, :, 0]

  tmp = []

  for j in range(0, masks_array.shape[1]):

    # Utworzenie maski
    mask = (masks_array[i][j] * depth_img)
  
    # Obliczanie momentów
    moments = cv2.moments(mask)

    # Wysokość maski
    y_indices, x_indices = np.where(mask > 0)
    if len(y_indices) == 0 or len(x_indices) == 0:
      tmp.extend([0, 0, 0,])
      continue
    mask_height = np.max(y_indices) - np.min(y_indices) + 1
    mask_width = np.max(x_indices) - np.min(x_indices) + 1

    # Momenty centralne
    m10 = moments['m10']
    m01 = moments['m01']
    m00 = moments['m00']

    # Środki masy
    if m00 == 0:
      tmp.extend([0, 0, 0,])
    else:
      x = m10/m00
      y = m01/m00
      z = np.mean(mask[mask > 0])
      tmp.extend([x, y, z])
  xyz_array.append(tmp)

xyz_array = np.array(xyz_array)
print(f"xyz_array shape -> {xyz_array.shape}")

xyz_array shape -> (172, 9)


In [5]:
df.head()

,image_id,source,ESJoint1,ESJoint2,ESJoint3,ESJoint4,ESJoint5,ESJoint6,gripper_finger_1_joint,gripper_finger_2_joint,color_path,depth_path
0,0,icm_111,3.848058,0.729946,1.842714,1.702291,0.986970,3.103531,0.603365,0.899934,/home/ruszczka/projekty/test_files/img/icm_111...,/home/ruszczka/projekty/test_files/img/icm_111...
1,1,icm_111,3.847902,0.730181,1.842891,1.702339,0.986970,3.103531,0.603365,0.899934,/home/ruszczka/projekty/test_files/img/icm_111...,/home/ruszczka/projekty/test_files/img/icm_111...
2,2,icm_111,3.847784,0.730299,1.842949,1.702291,0.987018,3.103483,0.603365,0.899934,/home/ruszczka/projekty/test_files/img/icm_111...,/home/ruszczka/projekty/test_files/img/icm_111...
3,3,icm_111,3.847725,0.730377,1.842969,1.702291,0.987018,3.103339,0.603365,0.899934,/home/ruszczka/projekty/test_files/img/icm_111...,/home/ruszczka/projekty/test_files/img/icm_111...
4,4,icm_111,3.847686,0.730416,1.842989,1.702291,0.987114,3.103291,0.603365,0.899934,/home/ruszczka/projekty/test_files/img/icm_111...,/home/ruszczka/projekty/test_files/img/icm_111...


In [6]:
model_cols = ['ESJoint1',	'ESJoint2',	'ESJoint3',	'ESJoint4',	'ESJoint5',	'ESJoint6',
              'gripper_finger_1_joint',	'gripper_finger_2_joint']
df_array = df[model_cols].to_numpy()
df_array.shape

(172, 8)

połączenie całego przetworzonego zbioru danych

In [7]:
data = np.concatenate((xyz_array, df_array), axis=1)
data.shape

(172, 17)

Usunięcie danych, gdzie nie wykryto maski

In [8]:
# delete incomplete_detect
data = np.delete(data, incomplete_detect, axis=0)
data.shape

(126, 17)

Utworzenie zbioru danych do uczenia - pary pozycji

In [ ]:
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

k = 62
robot_coords = data[:, 9:17]
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(robot_coords)

ds = []
for i in tqdm(range(data.shape[0])):
    distances, indices = nbrs.kneighbors([robot_coords[i]])
    # indices[0][1:] - bo pierwszy to te same współrzędne
    for j in indices[0][1:]:
        ds.append(np.concatenate([
            data[i, 0:9] - data[j, 0:9],  # X - różnica (Δ) współrzędnych x,y,z dla 3 elementów
            data[i, 9:17],                # X - wartości na przegubach dla pierwszego punktu
            data[i, 9:17] - data[j, 9:17] # Y - różnica (Δ) wartości na przegubach
        ]))
print(len(ds))
ds = np.array(ds)
print(ds.shape)
print(ds[0])

y_cols =[
  'x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z3',
  'J1', 'J2', 'J3', 'J4', 'J5', 'J6', 'G1', 'G2',
  'dJ1', 'dJ2', 'dJ3', 'dJ4', 'dJ5', 'dJ6', 'dG1', 'dG2'
]
df_training = pd.DataFrame(ds, columns=y_cols)
df_training.to_csv(str(SAVE_DIR)+'/df_training.csv', index=False)

100%|██████████| 126/126 [00:00<00:00, 800.63it/s]


7812
(7812, 25)
[ 8.87935516e-01 -5.47134821e-01 -1.71423340e+00  2.45226331e-01
  1.68362086e-01 -9.46197510e-02  6.87337319e-02 -3.65825385e-01
 -4.08859253e-01  3.42310500e+00  1.23096900e+00  1.93243200e+00
  1.52780100e+00  1.58762000e+00  2.66337400e+00  6.03365000e-01
  9.05047000e-01 -2.94000000e-04  5.88000000e-04  2.94000000e-04
 -5.27000000e-04  1.72600000e-03 -2.20500000e-03 -5.11300000e-03
  0.00000000e+00]


In [10]:
df_training.describe()

,x1,y1,z1,x2,y2,z2,x3,y3,z3,J1,...,G1,G2,dJ1,dJ2,dJ3,dJ4,dJ5,dJ6,dG1,dG2
count,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,...,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000
mean,6.906355,-4.275235,1.741325,5.930189,-4.672820,2.534143,6.248778,-4.113005,0.979704,3.421363,...,0.454593,1.068063,0.022801,-0.007904,-0.011898,0.019970,-0.002081,0.070984,-0.004035,0.004763
std,79.091944,50.492635,22.292466,83.793908,54.519223,28.104693,77.192940,54.527433,24.679641,0.292451,...,0.248917,0.270848,0.294785,0.305051,0.359851,0.197252,0.010423,0.488675,0.300597,0.327575
min,-343.605895,-186.558296,-71.893921,-359.909406,-198.088464,-84.633408,-285.441069,-201.526867,-76.135376,2.903283,...,0.178964,0.715857,-0.993491,-0.964975,-1.204937,-0.640533,-0.037774,-1.288736,-0.618705,-0.649384
25%,-47.199567,-38.531020,-13.341095,-51.408476,-41.251445,-16.793411,-47.323666,-40.131804,-15.600636,3.180270,...,0.189191,0.894821,-0.176679,-0.222692,-0.248568,-0.113323,-0.006568,-0.264755,-0.184078,-0.173851
50%,6.950635,-4.146958,2.009407,5.720972,-4.457911,2.069679,5.554745,-4.236944,1.166435,3.396005,...,0.593138,0.899934,0.020911,-0.008447,0.003459,0.017233,-0.000479,0.029985,0.000000,0.000000
75%,60.648161,30.016529,17.010063,62.378314,31.763645,22.065178,58.469885,31.805309,17.560612,3.668976,...,0.608478,1.360128,0.224201,0.210474,0.238975,0.152344,0.004266,0.355297,0.178964,0.173851
max,277.942309,151.280222,71.893921,324.646864,164.976197,84.633408,281.170353,201.526867,76.135376,4.002371,...,0.802782,1.370354,1.021880,0.905377,1.181636,0.648778,0.030440,1.659911,0.618705,0.654497
